In [ ]:
import sys, os
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.data import load_and_engineer, load_test_and_rul
from src.models import MeanRULBaseline, make_ridge_pipeline, make_random_forest, make_gradient_boosting
from src.metrics import rmse, nasa_score

In [ ]:
train_df = load_and_engineer()
X_test_df, y_test = load_test_and_rul()

non_feature_cols = ["engine_id", "cycle", "RUL", "RUL_clipped"]
feature_cols = [c for c in train_df.columns if c not in non_feature_cols]

X_train = train_df[feature_cols]
y_train = train_df["RUL_clipped"]
X_test = X_test_df[feature_cols]

print(f"Training set: {X_train.shape[0]} rows across {train_df['engine_id'].nunique()} engines")
print(f"Test set:     {X_test.shape[0]} engines (one prediction per engine)")
print(f"y_test range: {y_test.min()} to {y_test.max()}, mean {y_test.mean():.1f}")

In [ ]:
models = {
    "Baseline":         MeanRULBaseline(),
    "Ridge":            make_ridge_pipeline(alpha=1.0),
    "RandomForest":     make_random_forest(n_estimators=100),
    "GradientBoosting": make_gradient_boosting(),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "RMSE": rmse(y_test, y_pred),
        "NASA": nasa_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))